# FLUX.1-dev + uncensored LoRA: reference generation

This is a **separate reference experiment**, not the browser/WebGPU port. First validate the model and adapter with Diffusers on a CUDA runtime. FLUX.1-dev is gated on Hugging Face; accept its access conditions first. The notebook uses descriptive prose prompts.

The base model is under the FLUX.1-dev Non-Commercial License. Check the base and adapter model cards before use. Fictional adults only; no minors or non-consensual real people.

In [ ]:
import torch, platform
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('System RAM check is available through Colab runtime metrics.')


## Install Diffusers stack

Run this once per fresh runtime.

In [ ]:
%pip -q install -U diffusers transformers accelerate sentencepiece safetensors peft

## Authenticate

Create a Hugging Face read token, accept the FLUX.1-dev access conditions on its model page, and enter the token below. Do not paste the token into a shared notebook or commit it.

In [ ]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass('Hugging Face read token: ')
login(token=hf_token, add_to_git_credential=False)
del hf_token


## Load FLUX.1-dev and the adult-content LoRA

This is intentionally a conservative first pass. Sequential CPU offload lowers peak VRAM at the cost of speed. If the runtime runs out of host RAM or is impractically slow, stop here and choose a quantized/offload strategy rather than repeatedly downloading weights.

In [ ]:
import torch
from diffusers import FluxPipeline

BASE_MODEL = 'black-forest-labs/FLUX.1-dev'
LORA_REPO = 'whrw/Flux-Uncensored-V2'

pipe = FluxPipeline.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
)
pipe.load_lora_weights(LORA_REPO)

if torch.cuda.is_available():
    pipe.enable_sequential_cpu_offload()
else:
    raise RuntimeError('A CUDA runtime is required for this initial reference test.')

print('Pipeline and LoRA loaded.')


## Generate a reference image

Use a prose prompt describing an unmistakably adult fictional character. Keep the same prompt, seed, dimensions, and steps when comparing later implementations.

In [ ]:
import time
from IPython.display import display

prompt = (
    'A beautifully illustrated adult woman dressed as a gothic witch, wearing an elaborate '
    'crimson velvet dress with delicate black lace and an oversized pointed hat. She holds '
    'a glowing red apple in one hand. Dark gothic architecture in the background, rich '
    'cinematic lighting, soft atmospheric shadows, intricate fabric texture, expressive '
    'anime-inspired character design, carefully composed full illustration.'
)
seed = 123456789
generator = torch.Generator(device='cpu').manual_seed(seed)

started = time.perf_counter()
result = pipe(
    prompt=prompt,
    width=1024,
    height=1024,
    num_inference_steps=28,
    guidance_scale=3.5,
    max_sequence_length=512,
    generator=generator,
)
image = result.images[0]
elapsed = time.perf_counter() - started
image.save('flux-dev-uncensored-reference.png')
print(f'Generation time: {elapsed:.1f} s | seed: {seed} | 1024x1024 | 28 steps')
display(image)


## Record the result

If generation succeeds, record the runtime GPU, peak memory, elapsed time, and image quality. This is the reference target for any later quantized or browser implementation. Do not assume the browser can run this model: FLUX has a different and substantially larger architecture than SDXL.

In [ ]:
if torch.cuda.is_available():
    print('Peak allocated VRAM GiB:', round(torch.cuda.max_memory_allocated() / 1024**3, 2))
    print('Peak reserved VRAM GiB:', round(torch.cuda.max_memory_reserved() / 1024**3, 2))
print('Saved: flux-dev-uncensored-reference.png')
